# Phase 2: feature engineering
Run `python -m src.feature_engineering` from the project root first. This notebook transforms validation inputs only and leaves the saved splits unchanged.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import joblib
import numpy as np
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'src/feature_engineering.py').is_file():
    raise RuntimeError('Open from the project root or notebooks directory')
sys.path.insert(0, str(root))
from src.data_preprocessing import load_dataset, clean_dataset, make_features
from src.feature_engineering import PerovskiteFeatureEngineer
data_path = root.parent / 'Solar Cell data set2.csv'
audit = json.loads((root / 'data/processed/phase2/feature_audit.json').read_text())
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == audit['dataset_sha256']
clean, _, _ = clean_dataset(load_dataset(data_path))
X, _ = make_features(clean)
indices = np.load(root / 'data/processed/phase1/split_indices.npz')
engineer = joblib.load(root / 'data/processed/phase2/feature_engineer.joblib')
validation_features = engineer.transform(X.iloc[indices['validation']])
print(validation_features.shape)
display(validation_features.filter(regex='xai__.*(?:layer_count|unitless_mean)$').head())

Raw values and engineered features coexist. Bare numbers have unspecified units; SHAP or feature importance in later phases must not imply causal effects.